In [1]:
# Cell 1: Import all the libraries we need
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from fredapi import Fred
import sys
import os

In [2]:
# Cell 2: Load API key from config.py and connect to FRED
sys.path.append('..')  
from config import FRED_API_KEY

# Connect to FRED using API key
fred = Fred(api_key=FRED_API_KEY)

print("Connected to FRED successfully!")

Connected to FRED successfully!


In [3]:
# Cell 3: Download economic data from FRED
# Brent Crude Oil price (USD per barrel) - key driver of energy inflation
oil = fred.get_series('DCOILBRENTEU', observation_start='2000-01-01')

# GBP/USD exchange rate - affects import prices, which feed into inflation
gbp_usd = fred.get_series('DEXUSUK', observation_start='2000-01-01')

# US CPI - useful leading indicator since US data releases before UK
us_cpi = fred.get_series('CPIAUCSL', observation_start='2000-01-01')

print(f"Oil price observations: {len(oil)}")
print(f"GBP/USD observations: {len(gbp_usd)}")
print(f"US CPI observations: {len(us_cpi)}")

print("\nLatest oil price:", round(oil.iloc[-1], 2), "USD/barrel")
print("Latest GBP/USD rate:", round(gbp_usd.iloc[-1], 4))

Oil price observations: 6916
GBP/USD observations: 6914
US CPI observations: 317

Latest oil price: 69.56 USD/barrel
Latest GBP/USD rate: 1.3364


In [4]:
# Cell 4: Preview data
print("=== OIL PRICE (last 5 observations) ===")
print(oil.tail())

print("\n=== GBP/USD RATE (last 5 observations) ===")
print(gbp_usd.tail())

=== OIL PRICE (last 5 observations) ===
2026-06-30    70.46
2026-07-01    69.24
2026-07-02    68.53
2026-07-03    68.68
2026-07-06    69.56
dtype: float64

=== GBP/USD RATE (last 5 observations) ===
2026-06-26    1.3223
2026-06-29    1.3255
2026-06-30    1.3259
2026-07-01    1.3280
2026-07-02    1.3364
dtype: float64


In [5]:
# Cell 5: Save raw data to data/raw/ folder
oil.to_csv('../data/raw/brent_crude.csv', header=['oil_price'])
gbp_usd.to_csv('../data/raw/gbp_usd.csv', header=['gbp_usd'])
us_cpi.to_csv('../data/raw/us_cpi.csv', header=['us_cpi'])

print("\nAll raw data saved to data/raw/ folder!")


All raw data saved to data/raw/ folder!


In [6]:
# Let's see what sheets are inside the mm23 Excel file
import openpyxl

wb = openpyxl.load_workbook('../data/raw/mm23.xlsx', read_only=True)
print("Sheets in mm23:")
for name in wb.sheetnames:
    print(" -", name)

Sheets in mm23:
 - Contents
 - Table 1
 - Table 2
 - Table 3
 - Table 4
 - Table 5a, 5b
 - Table 6a, 6b, 6c
 - Table 7
 - Table 8
 - Table 9
 - Table 10
 - Table 11
 - Table 12
 - Table 13
 - Table 14
 - Table 15a, 15b, 15c
 - Table 16
 - Table 17
 - Table 18
 - Table 19
 - Table 20
 - Table 21
 - Table 22
 - Table 23
 - Table 24
 - Table 25
 - Table 26
 - Table 27
 - Table 28
 - Table 29
 - Table 30
 - Table 31a, 31b
 - Table 32
 - Table 33
 - Table 34
 - Table 35
 - Table 36a, 36b
 - Table 37
 - Table 38
 - Table 39
 - Table 40


In [8]:
# Load the correct sheet from mm23 - it's called 'Table 15a, 15b, 15c'
table15 = pd.read_excel('../data/raw/mm23.xlsx', sheet_name='Table 15a, 15b, 15c', header=None)

# Also define the months list which is needed later
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

In [9]:
# Only use rows 43 to 72 - the annual % change section
cpi_rows = table15.iloc[43:73].copy()

# Keep only rows where column 1 is a year
cpi_rows = cpi_rows[pd.to_numeric(cpi_rows[1], errors='coerce').notna()]

records = []
for _, row in cpi_rows.iterrows():
    year = int(row[1])
    for i, month in enumerate(months):
        value = row[i + 3]
        if pd.isna(value) or str(value).strip() == '..':
            pass  # future or missing - skip
        elif str(value).strip() == '-':
            date = pd.to_datetime(f'{year} {month}', format='%Y %b')
            records.append({'date': date, 'cpi': 0.0})
        else:
            try:
                date = pd.to_datetime(f'{year} {month}', format='%Y %b')
                records.append({'date': date, 'cpi': float(value)})
            except:
                pass

uk_cpi = pd.DataFrame(records).set_index('date').sort_index()

print(f"CPI observations: {len(uk_cpi)}")
print(f"Date range: {uk_cpi.index[0].date()} to {uk_cpi.index[-1].date()}")
print("\nLatest 5 readings:")
print(uk_cpi.tail())

CPI observations: 341
Date range: 1998-01-01 to 2026-05-01

Latest 5 readings:
            cpi
date           
2026-01-01  3.0
2026-02-01  3.0
2026-03-01  3.3
2026-04-01  2.8
2026-05-01  2.8


In [10]:
# Save clean CPI data
uk_cpi.to_csv('../data/raw/uk_cpi.csv')
print("CPI saved!")

CPI saved!


In [11]:
# UK average earnings index from FRED
uk_wages_fred = fred.get_series('LCEAMN01GBM661S', observation_start='2000-01-01')

# Convert index to annual % change (year on year)
uk_wages = uk_wages_fred.pct_change(12) * 100
uk_wages = uk_wages.dropna()

# Save to raw folder
uk_wages.to_csv('../data/raw/uk_wages.csv', header=['wages'])

print(f"Wages observations: {len(uk_wages)}")
print(f"Date range: {uk_wages.index[0].date()} to {uk_wages.index[-1].date()}")
print("\nLatest 5 readings:")
print(uk_wages.tail())

Wages observations: 303
Date range: 2001-01-01 to 2026-03-01

Latest 5 readings:
2025-11-01    4.031898
2025-12-01    4.950537
2026-01-01    3.676132
2026-02-01    3.792774
2026-03-01    3.112165
dtype: float64


In [14]:
# Save wages and unemployment to raw folder
uk_wages.to_csv('../data/raw/uk_wages.csv', header=['wages'])
uk_unemp_fred.to_csv('../data/raw/uk_unemployment.csv', header=['unemployment'])

print("Wages and unemployment saved!")

print("\n=== DATA COLLECTION COMPLETE ===")
print(f"UK CPI:          {len(uk_cpi)} monthly observations, up to {uk_cpi.index[-1].date()}")
print(f"UK Wages:        {len(uk_wages)} monthly observations, up to {uk_wages.index[-1].date()}")
print(f"UK Unemployment: {len(uk_unemp_fred)} monthly observations, up to {uk_unemp_fred.index[-1].date()}")
print(f"Brent Crude Oil: {len(oil)} daily observations, up to {oil.index[-1].date()}")
print(f"GBP/USD:         {len(gbp_usd)} daily observations, up to {gbp_usd.index[-1].date()}")
print(f"US CPI:          {len(us_cpi)} monthly observations, up to {us_cpi.index[-1].date()}")

Wages and unemployment saved!

=== DATA COLLECTION COMPLETE ===
UK CPI:          341 monthly observations, up to 2026-05-01
UK Wages:        303 monthly observations, up to 2026-03-01
UK Unemployment: 313 monthly observations, up to 2026-01-01
Brent Crude Oil: 6916 daily observations, up to 2026-07-06
GBP/USD:         6914 daily observations, up to 2026-07-02
US CPI:          317 monthly observations, up to 2026-05-01
